## Importing Libraries

In [1]:
import numpy as np

Using Cornell Movie dialog corpous for the model

Access corpous at https://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html

### Data Cleaning

In [176]:
data=[]
with open('movie_lines.txt', 'rb') as f:
    for line in f:
        data.append(line.decode(errors='ignore'))

In [177]:
data=[x.split('+++$+++')[-1].replace('\n','') for x in data]

In [178]:
len(data)

304713

In [179]:
#considering only first 3000 sentences and joining them to form a single paragraph.
data=data[:3000]
data=''.join(data)

In [180]:
#cleaning based on special characters (excluding dots and spaces)
def clean(string):
    string=string.lower()
    for i in range(len(string)-1):
        if not string[i].isalnum() and not string[i].isspace() and string[i]!='.':
            string=string[:i]+' '+string[i+1:]
    return string

In [205]:
#data tokenization based on spaces and dots. (decimal dots are also taken care of)
def tokenize(string):
    tokens=string.split()
    all_tok=[]
    for i in tokens:
        s_tok=i.split('.')
        
        #if there is no dot
        if(len(s_tok)==1):
            all_tok+=[i]
            continue
        
        #removing empty chars (in cases where words end with dot)
        s_tok=list(filter(('').__ne__,s_tok))
        
        #check if its a string or decimal dot
        flag=True
        for j in s_tok:
            if not j.isdigit():
                flag=False
                break
        #if dot in strings
        if flag==False:
            all_tok+=[u for x in s_tok for u in (x, '.')]
        
        #if decimal dot, append complete decimal as token
        else:
            all_tok+=[i]
    return all_tok

In [206]:
tokenize('hi. hel 34.60 He'.lower())

['hi', '.', 'hel', '34.60', 'he']

In [183]:
c_data=clean(data)

In [207]:
# create a dictionary of unique words in the corpus
word_dct= list(set(tokenize(c_data)))
# word_dct.remove("")
len(word_dct)

3594

In [208]:
word_dct[:10]

['makin',
 'ha',
 'my',
 'maneuver',
 'movie',
 'here',
 'homesick',
 'protecting',
 'titles',
 '4']

In [209]:
#reverse dictionary for mapping word to indices
rev_dct = {j:i for i,j in enumerate(word_dct)}

#checking the index of a random word from corpus
rev_dct['.'],rev_dct['ladder']

(322, 2606)

In [210]:
w_count=len(word_dct)

In [211]:
#2-d to store count of next word based on current word
uni_count=np.zeros((w_count,w_count))

#count of current word
word_count=np.zeros(w_count)

#tokenisation based on spaces and special characters with decimal being taken care of
all_words=list(tokenize(c_data))
# while '' in all_words: all_words.remove('')
all_words=list(filter(('').__ne__, all_words))

In [213]:
#iterating throughout the complete corpus to store the count
for i in range(0,len(all_words)-1):
    prev_index=rev_dct[all_words[i]]
    word_count[prev_index]+=1
    cur_index =rev_dct[all_words[i+1]]
    uni_count[prev_index,cur_index]+=1

word_count[rev_dct[all_words[-1]]]+=1

## Probability of next word

Formula to calculate the probability of next word W’ if the current word is W in the corpus is

$$ P(W'|W) = \frac{\text{Count}(W,W')}{\text{Count}(W)} $$

where, $ \text{Count}(W,W’)$ is number of times $W’$ follows $W$ and Count$(W)$ is total occurrence of $W$ in the
corpus.


In [214]:
#calculating probability matrix using word count and next word count using the formula
for i in range(len(uni_count)):
    uni_count[i]=uni_count[i]/word_count[i]

In [215]:
# np.savetxt("prob_matrix.csv", uni_count, delimiter=",")

In [221]:
#function to predict next words based on a single word.
def predict(word,limit=10):
    
    cur_word=word.lower()
    print (cur_word,end=' ')
    for i in range(limit):
        #index of next word using probability matrix
        cur_prob_idx = uni_count[rev_dct[cur_word]]
        
        nextW_idx = np.random.choice(np.argwhere(cur_prob_idx == np.amax(cur_prob_idx)).ravel())
        #index to word using dictionary of words
        nextW=word_dct[nextW_idx]
        cur_word=nextW
        print (nextW,end=' ')
        if(nextW=='.'):
            break
    print ()

In [253]:
seeds=['consider','machine','particular','shell','lung','drug','750','desires','aristotle','takin']
for i in seeds: predict(i)

consider heresy don t know . 
machine programmed to be a little more than that s a 
particular care of the money . 
shell out of the money . 
lung cancer issue number one . 
drug related to be a little more than that s a 
750 leagues or not a little moody . 
desires his girl . 
aristotle erathostene ptolemeus yes . 
takin care of the money . 
